In [13]:
import pandas as pd
import numpy as np
import openpyxl
import os
import routes as gv
import matplotlib.pyplot as plt
import pathlib as Path
import re
from typing import Optional, Callable,Any

In [ ]:

# df.iloc[fila, columna] -> Accede al valor por índice de fila y columna (0-indexed)
# df["Columna"].iloc[fila] -> Accede al valor por nombre de columna y posición de fila
# df.iat[fila, columna] -> Acceso rápido a un solo valor por índice de fila y columna
#drop() elimina una columan
#banks=lambda x:x["banks"].astype("Int32")  es uan funciona anonima para declarar el tipo de datos de la columna 
#Float32
#Prueba con decorador fallida complica mas las cosas !!!!°!
#Prueba con decorador para asignar nombres a a las columnas 
# def endulcoradorparacambiarnombreytipodecolumna(nombreColumna,posciionColumna,tipoDato):

#     def endulcorador1(funcion):
#         @wraps(funcion) # Buenas prácticas: Preservar metadatos
#         def endulcorador2(*args, **kwargs):
            
#             # 1. Obtener el DataFrame de la función original
#             df = funcion(*args, **kwargs)
#             nombre_actual = df.columns[posciionColumna]
#             df_modificado = (
#                 df.rename(columns={nombre_actual: nombreColumna})

#                 .assign(**{
#                     nombreColumna: lambda x: pd.to_numeric(x[nombreColumna], errors='coerce').astype(tipoDato)
#                 })
#             )
#             return df_modificado
            
#         return endulcorador2
#     return endulcorador1


# @endulcoradorparacambiarnombreytipodecolumna("miguelito",1,"Float32")
class dataFrameLeo:
    def __init__(self,ruta):
        self.df=pd.read_excel(ruta, engine="openpyxl")
        self.nombrearchivo=ruta.name

def limpiar_datos_de_carpeta_040_12d_R1(rutaArchivo, 
                                        transformacion_adicional: Optional[Callable] = None):

    df = pd.read_excel(rutaArchivo, engine="openpyxl")  # usa engine="xlrd" si es .xls


    df_limpio=df.drop(columns=["Unnamed: 0"]) \
                .rename(columns={"Banco":"banks"})\
                .assign(year=str(df.columns[2])[:4],
                        month=str(df.columns[2])[-2:],)\
                .assign(
#asignacion de tipo de dato de columnas


                        year=lambda x:x["year"].astype("Int32"),
                        month=lambda x:x["month"].astype("Int32"),
                        banks=lambda x:x["banks"].astype(str),)\
                .iloc[:-1]
    
    if transformacion_adicional:
        # Usamos .pipe() para integrar la función adicional en la cadena.
        # Esto le da el DataFrame df_limpio como primer argumento.
        df_limpio = df_limpio.pipe(transformacion_adicional)

    return df_limpio




#Esta funcion va a retornar una Dataframe juntando todo el contenido preprocesado con la funcion de limpieza
def limpiar_datos_de_carpeta(funcionlimpieza,rutarchivo,**kwargs):
    df_unido=pd.DataFrame()

    for archivo in rutarchivo:
        k=funcionlimpieza(archivo,**kwargs)
        df_unido=pd.concat([df_unido,k],ignore_index=True)
    
    return df_unido
###################

def pipeline(*args: Callable) -> Callable:

    def salida(datos_iniciales: Any):
        resultado = datos_iniciales
        
        for func in args:
            resultado = func(resultado) 
            
        return resultado
    return salida

def asignar_y_tipificar(df, nombre_columna_nuevo, posicion_columna, tipo_dato):
 
    nombre_actual = df.columns[posicion_columna]
    df=df.rename(columns={nombre_actual:nombre_columna_nuevo}



    ).assign()


    return (
        df.rename(columns={nombre_actual: nombre_columna_nuevo})
        .assign(**{
            nombre_columna_nuevo: lambda x: pd.to_numeric(x[nombre_columna_nuevo], errors='coerce').astype(tipo_dato)
        })
    )

def transformar_monto(df):
    return asignar_y_tipificar(df, "quantyCreditCards", 1, "Int32")
            
# importantge no borrar------------------------------------------>

# Numero_de_tarjetas_de_credito_por_institucion = limpiar_datos_de_carpeta(
#     limpiar_datos_de_carpeta_040_12d_R1,
#     gv.ARCHIVOS_DENTRO_CARPETAS_OUTPUT[0],
#     transformacion_adicional=transformar_monto
# )
# importantge no borrar------------------------------------------>





#funcion paraobtenr las columans de año y mes acepata dos argumentos optionales que le pasemos la posicuion de la fecha si esta en la cabecera de una columna o por localizacion de indice en el excel
def crear_columna_year_month(df,indexcolumnafecha:Optional[callable]=None,ilocfecha:Optional[callable]=None):
    if indexcolumnafecha:
        df=df.assign(year=str(df.columns[indexcolumnafecha])[:4],
        month=str(df.columns[indexcolumnafecha])[-2:]
        ). assign(
            year=lambda x: x["year"].astype(np.int32),
            month=lambda x:x["month"].astype(np.int32)
        )
    elif ilocfecha:
        print("funcion sin definir")
    return df        

def limpiar_datos_de_carpeta_040_12d_R2(ruta,transformacion_adicional:Optional[callable]=None):
    rt=pd.read_excel(ruta,dtype={"Banco":str},usecols="B:C").iloc[:-1]
    rt=crear_columna_year_month(rt,1)
    if transformacion_adicional:
        rt = rt.pipe(transformacion_adicional)

    return rt



def abc(df):
   return asignar_y_tipificar(df,"creditCardBalanceByInstitution",1,"Float32")



unido=limpiar_datos_de_carpeta(limpiar_datos_de_carpeta_040_12d_R2,gv.ARCHIVOS_DENTRO_CARPETAS_OUTPUT[1],transformacion_adicional=abc).fillna(0)

unido.head(5)
#PRUEBAS
x=gv.ARCHIVOS_DENTRO_CARPETAS_OUTPUT[1][1]
x=limpiar_datos_de_carpeta_040_12d_R2(x)


In [ ]:
x.columns[0]
y=x.rename(columns={"Banco":"change"})
y.assign()

,change,201108,year,month
0,Banco Ualá,NaN,2011,8
1,Actinver,NaN,2011,8
2,Afirme,1.623377e+08,2011,8
3,American Express,6.577740e+09,2011,8
4,Banamex,NaN,2011,8
